# 06: Prediction Tracking
Reads the running log the daily script (`mets/scripts/predict_next_home_game.py`) appends to on every run, and lays it out as one row per game, one column per lead time (days before the game), so you can see how each prediction moved as more information (weather forecast, standings, probable starter) became available. Run this cell any time, it always reflects the latest log.

In [4]:
import pandas as pd
from pathlib import Path

LOG_PATH = Path.home() / "Library" / "Application Support" / "mets-prediction" / "data" / "mets_2026_predictions_log.csv"

log = pd.read_csv(LOG_PATH)
log["game_label"] = log["game_date"] + " vs " + log["opponent"]
print(f"{len(log)} logged run(s) across {log['game_label'].nunique()} game(s)")
log.tail()

36 logged run(s) across 6 game(s)


,run_date,game_date,opponent,predicted_attendance,mets_starter,mets_starter_era,opp_starter,opp_starter_era,avg_temp_f,avg_precip_mm,is_promo,days_out,actual_attendance,game_label
31,2026-08-11,2026-08-15,Washington Nationals,39829,NaN,NaN,NaN,NaN,77.7,0.0,0,4,NaN,2026-08-15 vs Washington Nationals
32,2026-08-11,2026-08-16,Washington Nationals,38825,NaN,NaN,NaN,NaN,77.3,0.8,0,5,NaN,2026-08-16 vs Washington Nationals
33,2026-08-11,2026-08-17,San Diego Padres,37459,NaN,NaN,NaN,NaN,85.5,0.0,0,6,NaN,2026-08-17 vs San Diego Padres
34,2026-08-11,2026-08-18,San Diego Padres,37666,NaN,NaN,NaN,NaN,83.7,0.0,0,7,NaN,2026-08-18 vs San Diego Padres
35,2026-08-11,2026-08-19,San Diego Padres,37636,NaN,NaN,NaN,NaN,83.5,0.0,0,8,NaN,2026-08-19 vs San Diego Padres


## Prediction by Lead Time
One row per game, one column per days-out value the daily script actually ran on (so the columns fill in over the days leading up to each game). `Actual` and `% Error` are blank until you fill in `actual_attendance` in the log CSV after the game is played, comparing the *closest-to-game-day* prediction against the real number.

In [5]:
pivot = log.pivot_table(index="game_label", columns="days_out", values="predicted_attendance", aggfunc="last")
pivot = pivot.reindex(sorted(pivot.columns, reverse=True), axis=1)
pivot.columns = [f"{int(c)}d out" for c in pivot.columns]

game_dates = log.groupby("game_label")["game_date"].last()
actuals = log.groupby("game_label")["actual_attendance"].last()

# The prediction closest to game day (smallest days_out) is treated as the "final" call
closest_pred = log.sort_values("days_out").groupby("game_label")["predicted_attendance"].first()

pivot = pivot.join(game_dates).join(actuals.rename("Actual"))
pivot["% Error"] = ((closest_pred - pivot["Actual"]).abs() / pivot["Actual"] * 100).round(1)
pivot = pivot.sort_values("game_date").drop(columns="game_date")

pivot

,9d out,8d out,7d out,6d out,5d out,4d out,3d out,2d out,Actual,% Error
game_label,,,,,,,,,,
2026-08-14 vs Washington Nationals,35240.0,36541.0,38144.0,38144.0,38646.0,37726.0,37726.0,38519.0,NaN,NaN
2026-08-15 vs Washington Nationals,36652.0,38995.0,38995.0,39829.0,38677.0,39829.0,39318.0,NaN,NaN,NaN
2026-08-16 vs Washington Nationals,38665.0,39261.0,38944.0,39499.0,38825.0,38314.0,NaN,NaN,NaN,NaN
2026-08-17 vs San Diego Padres,35902.0,37421.0,37421.0,37459.0,37459.0,NaN,NaN,NaN,NaN,NaN
2026-08-18 vs San Diego Padres,37627.0,37627.0,37666.0,37627.0,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-19 vs San Diego Padres,37284.0,37636.0,37636.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Accuracy by Lead Time
Once a few games have `actual_attendance` filled in, this breaks out mean error by how far ahead of the game the prediction was made, the core question: does the prediction actually get better as game day approaches?

In [6]:
scored = log.dropna(subset=["actual_attendance"]).copy()

if scored.empty:
    print("No games with actual_attendance filled in yet, nothing to score.")
else:
    scored["abs_error"] = (scored["predicted_attendance"] - scored["actual_attendance"]).abs()
    scored["abs_pct_error"] = scored["abs_error"] / scored["actual_attendance"] * 100

    def bucket(days_out):
        if days_out >= 14:
            return "14+ days out"
        if days_out >= 7:
            return "7-13 days out"
        if days_out >= 3:
            return "3-6 days out"
        return "0-2 days out"

    scored["lead_time"] = scored["days_out"].apply(bucket)
    bucket_order = ["14+ days out", "7-13 days out", "3-6 days out", "0-2 days out"]
    accuracy = scored.groupby("lead_time").agg(
        n=("abs_error", "count"),
        mae=("abs_error", "mean"),
        mape=("abs_pct_error", "mean"),
    ).reindex(bucket_order).dropna(how="all")
    display(accuracy.round(1))

No games with actual_attendance filled in yet, nothing to score.


**Next:** monitor this notebook as the automated daily predictions come in over the remaining 2026 home games.